# Phase 3 — Match Logic Publisher Check

This notebook is a Phase 3 helper for checking MQTT match-event flow.

Scope:
- Connect to configured MQTT broker
- Subscribe to match events
- Print incoming match events for verification

In later phases, this notebook can evolve into full match-logic processing.

In [1]:
import json
from pathlib import Path
import sys

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

In [2]:
if "connector" in globals() and connector is not None:
    try:
        connector.disconnect()
    except Exception:
        pass

cfg = load_config()
topic_match = f"{cfg.mqtt.base_topic}/events/match"

connector = MqttConnector(cfg.mqtt, client_id_suffix="agent-match-logic")
connector.connect()
is_connected = connector.wait_for_connection(timeout=3.0)

if not is_connected:
    print(f"Could not confirm MQTT connection to {cfg.mqtt.host}:{cfg.mqtt.port}")
else:
    def on_message(client, userdata, message):
        try:
            payload = json.loads(message.payload.decode("utf-8"))
        except Exception:
            payload = message.payload.decode("utf-8", errors="replace")
        print(f"[match-event] topic={message.topic} payload={payload}")

    connector.client.on_message = on_message
    connector.client.subscribe(topic_match)
    print(f"Connected to {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Subscribed to: {topic_match}")

Connected to 127.0.0.1:1883
Subscribed to: simulated-city/events/match


In [ ]:
# Run this cell when finished
connector.disconnect()